<a href="https://colab.research.google.com/github/LevramUzo/gracce-groundwater-nigeria/blob/main/notebooks/01_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PHASE 1: RE-INITIALIZATION & BOUNDARY TESTING OVER NIGERIA
# ==============================================================================
import ee
from google.colab import userdata

try:
    # 1. Fetch credentials securely from the local environment vault
    PROJECT_ID = userdata.get('GEE_PROJECT_ID')

    # 2. Authenticate the active workspace session
    print("Connecting to Google Cloud Ecosystem... Allow permission via pop-up.")
    ee.Authenticate()

    # 3. Mount the engine to your cloud profile
    ee.Initialize(project=PROJECT_ID)
    print(f"\n[SUCCESS] Google Earth Engine linked securely using project: {PROJECT_ID}")

    # 4. Load the Global International Boundaries database and isolate Nigeria
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria_boundary = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # 5. Force the cloud geometry area calculation to process instantly
    nigeria_area_m2 = nigeria_boundary.geometry().area(maxError=1)
    nigeria_area_km2 = nigeria_area_m2.divide(1e6).getInfo()

    print("\n" + "="*55)
    print(f"NATIONAL STUDY AREA BOUNDARY ACTIVE")
    print("="*55)
    print(f"Target Boundary Region : Federal Republic of Nigeria")
    print(f"Calculated Basin Area  : {nigeria_area_km2:,.2f} km²")
    print("="*55 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Setup failed at re-initialization.")
    print(f"Details: {e}")


Connecting to Google Cloud Ecosystem... Allow permission via pop-up.


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python



[SUCCESS] Google Earth Engine linked securely using project: weather-app-492314

NATIONAL STUDY AREA BOUNDARY ACTIVE
Target Boundary Region : Federal Republic of Nigeria
Calculated Basin Area  : 912,554.10 km²



In [ ]:
# ==============================================================================
# STEP 2.2: HYDROCLIMATIC PREDICTOR EXTRACTION (CHIRPS & MODIS LST)
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Reference our Nigeria boundary layer from the first cell
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # Define our temporal research window (2003 - 2016)
    START_DATE = '2003-01-01'
    END_DATE = '2016-12-31'

    print("Step 1/3: Accessing Climate Catalogs inside Earth Engine...")

    # 2. Ingest the CHIRPS High-Resolution Precipitation Dataset
    chirps_collection = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate(START_DATE, END_DATE) \
        .select('precipitation')

    # 3. Ingest the MODIS Land Surface Temperature Dataset (Daytime)
    modis_lst_collection = ee.ImageCollection("MODIS/061/MOD11A1") \
        .filterDate(START_DATE, END_DATE) \
        .select('LST_Day_1km')

    # 4. Define an extraction function to process monthly blocks over Nigeria
    def extract_monthly_climate(year, month):
        # Establish month-specific date bounds
        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, 'month')

        # Calculate monthly totals for Rainfall and monthly averages for Temperature
        monthly_rain = chirps_collection.filterDate(start, end).sum()
        monthly_temp = modis_lst_collection.filterDate(start, end).mean()

        # Flatten pixels inside Nigeria down to national numeric values
        rain_val = monthly_rain.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=5500, maxPixels=1e9
        ).get('precipitation')

        temp_val = monthly_temp.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=5500, maxPixels=1e9
        ).get('LST_Day_1km')

        # Convert raw MODIS Kelvin values to readable Celsius degrees
        # (MODIS data is stored multiplied by a scale factor of 0.02)
        celsius_val = ee.Number(temp_val).multiply(0.02).subtract(273.15)

        # Return a structured record containing calendar keys
        date_str = start.format('YYYY-MM')
        return ee.Feature(None, {
            'date': date_str,
            'Rainfall_mm': rain_val,
            'LST_Celsius': ee.Algorithms.If(temp_val, celsius_val, None)
        })

    # 5. Build our timeline list loop to process through each year/month sequentially
    years = list(range(2003, 2017))
    months = list(range(1, 13))

    print("Step 2/3: Running spatial pixel reduction across Nigeria (This takes ~30 seconds)...")
    results_list = []

    for y in years:
        for m in months:
            # Execute cloud reduction commands for this month block
            feat = extract_monthly_climate(y, m)
            results_list.append(feat)

    # Resolve the cloud calculations and stream the clean text data back
    server_side_features = ee.FeatureCollection(results_list).getInfo()

    # 6. Reconstruct raw records into a highly scannable Data Frame
    compiled_records = [f['properties'] for f in server_side_features['features']]
    climate_df = pd.DataFrame(compiled_records)
    climate_df = climate_df.dropna().sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA CLIMATE MATRIX COMPILED")
    print("="*65)
    print(climate_df.head(12)) # Preview a full calendar year
    print("="*65)
    print(f"Total processed records: {len(climate_df)} calendar months.")
    print("="*65 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Climate feature extraction failed.")
    print(f"Details: {e}")


Step 1/3: Accessing Climate Catalogs inside Earth Engine...
Step 2/3: Running spatial pixel reduction across Nigeria (This takes ~30 seconds)...

SUCCESS: NIGERIA CLIMATE MATRIX COMPILED
    LST_Celsius  Rainfall_mm     date
0     32.862381     6.546161  2003-01
1     35.399531    11.000401  2003-02
2     35.070512    16.483187  2003-03
3     37.283007    74.405237  2003-04
4     37.878470   112.640571  2003-05
5     29.155469   188.931344  2003-06
6     29.091259   216.928714  2003-07
7     28.365416   253.560656  2003-08
8     29.168214   231.016700  2003-09
9     31.794775    96.118767  2003-10
10    32.622224    20.124457  2003-11
11    31.650024     2.803890  2003-12
Total processed records: 168 calendar months.



In [3]:
# ==============================================================================
# STEP 2.3: TARGET VARIABLE EXTRACTION (NASA GRACE TWSA OVER NIGERIA)
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Fetch Nigeria's border using the global administrative layer
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # Define our matching temporal timeline (2003 - 2016)
    START_DATE = '2003-01-01'
    END_DATE = '2016-12-31'

    print("Accessing NASA GRACE Satellite Gravity Grids...")

    # 2. Ingest the official NASA GRACE Land Mass Grid collection (RL06 CSR)
    grace_collection = ee.ImageCollection("NASA/GRACE/MASS_GRIDS_V04/LAND") \
        .filterDate(START_DATE, END_DATE) \
        .select('lwe_thickness_csr')

    # 3. Create a regional function to calculate average water weight anomalies
    def extract_monthly_grace(image):
        # Reduce global pixels down to a single average thickness number inside Nigeria
        mean_val = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=nigeria.geometry(),
            scale=25000, # Native resolution of the GRACE sensor footprint (25km)
            maxPixels=1e9
        ).get('lwe_thickness_csr')

        # Pull the year and month to match our climate table formatting
        date_str = image.date().format('YYYY-MM')

        return ee.Feature(None, {
            'date': date_str,
            'TWSA_mm': mean_val
        })

    print("Running geographic weight reductions across the Nigerian basin... Please wait.")

    # 4. Map the extraction loop across the historical timeline
    processed_features = grace_collection.map(extract_monthly_grace).getInfo()

    # 5. Build our structured data frame
    extracted_data = [f['properties'] for f in processed_features['features']]
    grace_df = pd.DataFrame(extracted_data)
    grace_df = grace_df.dropna().sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA GRACE TARGET ANOMALIES COMPILED")
    print("="*65)
    print(grace_df.head(12)) # Preview a full annual sequence
    print("="*65)
    print(f"Total compiled satellite observations: {len(grace_df)} months.")
    print("="*65 + "\n")

except Exception as e:
    print(f"\n[ERROR]: GRACE feature extraction failed.")
    print(f"Details: {e}")


Accessing NASA GRACE Satellite Gravity Grids...
Running geographic weight reductions across the Nigerian basin... Please wait.

SUCCESS: NIGERIA GRACE TARGET ANOMALIES COMPILED
     TWSA_mm     date
0  -0.091221  2003-01
1  -0.119114  2003-02
2  -0.133866  2003-03
3  -0.144336  2003-04
4  -0.013229  2003-06
5   0.055186  2003-07
6   0.144293  2003-08
7   0.133955  2003-09
8   0.070251  2003-10
9   0.017593  2003-11
10 -0.017884  2003-12
11 -0.061574  2004-02
Total compiled satellite observations: 150 months.



In [6]:
# ==============================================================================
# STEP 2.4: DATASET MERGING AND COORDINATION
# ==============================================================================
import pandas as pd

try:
    # Use pandas to link your climate data frame and GRACE data frame
    # We join them specifically on the 'date' column so everything lines up perfectly
    master_df = pd.merge(climate_df, grace_df, on='date', how='inner')

    # Sort the rows chronologically from past to present
    master_df = master_df.sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: MASTER HYDRO-MACHINE LEARNING DATASET MERGED")
    print("="*65)
    print(master_df.head(10)) # Preview a full lined-up year of records
    print("="*65)
    print(f"Total matching monthly rows coordinated: {len(master_df)} months.")
    print("="*65 + "\n")

    # Optional: Save a copy to your active Colab temporary files list
    master_df.to_csv('nigeria_groundwater_master.csv', index=False)
    print("[STATUS]: Master dataset saved locally as 'nigeria_groundwater_master.csv'")

except Exception as e:
    print(f"\n[ERROR]: Dataset merge failed.")
    print(f"Details: {e}")
    print("[FIX]: Make sure you ran both Cell 2 (Climate) and Cell 3 (GRACE) in this exact session.")



SUCCESS: MASTER HYDRO-MACHINE LEARNING DATASET MERGED
   LST_Celsius  Rainfall_mm     date   TWSA_mm
0    32.862381     6.546161  2003-01 -0.091221
1    35.399531    11.000401  2003-02 -0.119114
2    35.070512    16.483187  2003-03 -0.133866
3    37.283007    74.405237  2003-04 -0.144336
4    29.155469   188.931344  2003-06 -0.013229
5    29.091259   216.928714  2003-07  0.055186
6    28.365416   253.560656  2003-08  0.144293
7    29.168214   231.016700  2003-09  0.133955
8    31.794775    96.118767  2003-10  0.070251
9    32.622224    20.124457  2003-11  0.017593
Total matching monthly rows coordinated: 150 months.

[STATUS]: Master dataset saved locally as 'nigeria_groundwater_master.csv'


In [7]:
# ==============================================================================
# STEP 2.5: TOPOGRAPHIC PREDICTOR EXTRACTION (NASA SRTM ELEVATION & SLOPE)
# ==============================================================================
import ee

try:
    # 1. Fetch Nigeria vector boundaries
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    print("Accessing NASA Digital Elevation Models...")

    # 2. Load the official NASA SRTM 30m Global Elevation Dataset
    srtm_elevation = ee.Image("USGS/SRTMGL1_003")

    # 3. Calculate topographic slope mathematically from the elevation layer
    srtm_slope = ee.Terrain.slope(srtm_elevation)

    # 4. Flatten pixels across Nigeria down to regional national averages
    elev_val = srtm_elevation.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=1000, maxPixels=1e9
    ).get('elevation')

    slope_val = srtm_slope.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=1000, maxPixels=1e9
    ).get('slope')

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA TOPOGRAPHIC BASELINE EXTRACTED")
    print("="*65)
    print(f"Mean National Elevation : {elev_val.getInfo():,.2f} meters above sea level")
    print(f"Mean National Slope     : {slope_val.getInfo():,.2f} degrees steepness")
    print("="*65)
    print("[STATUS]: Static spatial terrain attributes initialized.")

except Exception as e:
    print(f"\n[ERROR]: Topographic extraction failed.")
    print(f"Details: {e}")


Accessing NASA Digital Elevation Models...

SUCCESS: NIGERIA TOPOGRAPHIC BASELINE EXTRACTED
Mean National Elevation : 330.24 meters above sea level
Mean National Slope     : 0.79 degrees steepness
[STATUS]: Static spatial terrain attributes initialized.


In [8]:
# ==============================================================================
# STEP 2.6: BROADCASTING STATIC TERRAIN VALUES TO THE MASTER MATRIX
# ==============================================================================
import pandas as pd

try:
    # 1. Read your terrain variables directly from your active variables
    mean_elevation = 330.24
    mean_slope = 0.79

    # 2. Assign these values as new columns across every single row of your matrix
    master_df['Elevation_m'] = mean_elevation
    master_df['Slope_deg'] = mean_slope

    # 3. Rearrange the columns to make the dataset beautiful and professional
    column_order = ['date', 'Rainfall_mm', 'LST_Celsius', 'Elevation_m', 'Slope_deg', 'TWSA_mm']
    final_ml_ready_df = master_df[column_order]

    print("\n" + "="*70)
    print("SUCCESS: FINAL HIGHER-LEVEL MACHINE LEARNING MATRIX COMPILED")
    print("="*70)
    print(final_ml_ready_df.head(12)) # Preview a full calendar year
    print("="*70)
    print(f"Total Rows: {len(final_ml_ready_df)} months | Total Columns: {len(final_ml_ready_df.columns)}")
    print("="*70 + "\n")

    # Save this ultimate table to your Colab directory
    final_ml_ready_df.to_csv('nigeria_groundwater_final_ml.csv', index=False)
    print("[STATUS]: Exported clean copy as 'nigeria_groundwater_final_ml.csv'")

except Exception as e:
    print(f"\n[ERROR]: Column broadcasting failed.")
    print(f"Details: {e}")




SUCCESS: FINAL HIGHER-LEVEL MACHINE LEARNING MATRIX COMPILED
       date  Rainfall_mm  LST_Celsius  Elevation_m  Slope_deg   TWSA_mm
0   2003-01     6.546161    32.862381       330.24       0.79 -0.091221
1   2003-02    11.000401    35.399531       330.24       0.79 -0.119114
2   2003-03    16.483187    35.070512       330.24       0.79 -0.133866
3   2003-04    74.405237    37.283007       330.24       0.79 -0.144336
4   2003-06   188.931344    29.155469       330.24       0.79 -0.013229
5   2003-07   216.928714    29.091259       330.24       0.79  0.055186
6   2003-08   253.560656    28.365416       330.24       0.79  0.144293
7   2003-09   231.016700    29.168214       330.24       0.79  0.133955
8   2003-10    96.118767    31.794775       330.24       0.79  0.070251
9   2003-11    20.124457    32.622224       330.24       0.79  0.017593
10  2003-12     2.803890    31.650024       330.24       0.79 -0.017884
11  2004-02     5.841645    32.621323       330.24       0.79 -0.061574
To

In [9]:
# ==============================================================================
# STEP 2.7: PIXEL-LEVEL MULTI-SENSOR SPATIAL EXTRACTION FOR NIGERIA
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Isolate the Nigeria boundary mask
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    print("[1/4] Loading planetary satellite grids...")
    # 2. Load the raw continuous terrain layers (Unique across space)
    srtm_elevation = ee.Image("USGS/SRTMGL1_003")
    srtm_slope = ee.Terrain.slope(srtm_elevation)

    # 3. Load a sample climate slice (e.g., peak rainy season July 2016)
    # to extract our initial spatial training pixels
    chirps_rain = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate('2016-07-01', '2016-07-31').sum()

    modis_lst = ee.ImageCollection("MODIS/061/MOD11A1") \
        .filterDate('2016-07-01', '2016-07-31').mean().select('LST_Day_1km')

    # 4. Combine all layers into a single multi-channel "Image Stack"
    # We convert raw LST Kelvin to clean Celsius inside the stack calculation
    lst_celsius = modis_lst.multiply(0.02).subtract(273.15).rename('LST_Celsius')

    image_stack = srtm_elevation.rename('Elevation_m') \
        .addBands(srtm_slope.rename('Slope_deg')) \
        .addBands(chirps_rain.rename('Rainfall_mm')) \
        .addBands(lst_celsius)

    print("[2/4] Generating a 0.05-degree coordinate sampling grid over Nigeria...")
    # Create a random cluster of spatial sample pixels inside Nigeria's borders
    # to build our localized machine learning training matrix
    spatial_samples = image_stack.sample(
        region=nigeria.geometry(),
        scale=5500, # 5500 meters matches our exact 0.05-degree target resolution!
        numPixels=500, # We start with 500 unique points across the country to keep it fast
        geometries=True # Keep the exact Latitude/Longitude coordinates for our GMT maps
    )

    print("[3/4] Streaming localized pixel rows from Google Cloud centers...")
    sample_data = spatial_samples.getInfo()['features']

    # 5. Unpack the cloud features into unique rows and columns
    pixel_records = []
    for f in sample_data:
        props = f['properties']
        coords = f['geometry']['coordinates']
        props['Longitude'] = coords[0]
        props['Latitude'] = coords[1]
        pixel_records.append(props)

    # 6. Compile into our final unique geographic database
    pixel_df = pd.DataFrame(pixel_records)
    pixel_df = pixel_df.dropna().reset_index(drop=True)

    print("\n" + "="*75)
    print("SUCCESS: PIXEL-BASED UNIQUE GEOSPATIAL MATRIX GENERATED")
    print("="*75)
    print(pixel_df.head(10)) # Preview the first 10 completely unique locations
    print("="*70)
    print(f"Total unique geographic pixel entries compiled: {len(pixel_df)} rows")
    print("="*75 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Spatial grid sampling failed.")
    print(f"Details: {e}")


[1/4] Loading planetary satellite grids...
[2/4] Generating a 0.05-degree coordinate sampling grid over Nigeria...
[3/4] Streaming localized pixel rows from Google Cloud centers...

SUCCESS: PIXEL-BASED UNIQUE GEOSPATIAL MATRIX GENERATED
   Elevation_m  LST_Celsius  Rainfall_mm  Slope_deg  Longitude   Latitude
0          119    29.870000   166.781202   0.167422   8.282639   7.541670
1          327    31.450000   156.728150   0.258080   3.877383  10.571777
2          363    32.726667   137.640795   0.036981  10.850450  12.186641
3          253    29.278000   252.652251   2.402760  12.521896   9.101273
4           79    29.660000   175.133596   0.109412   8.038870   8.189485
5          435    28.987500   234.526795   0.082209  13.450844  10.938003
6          149    26.790000   161.950132   0.618208   7.529010   8.175558
7          356    31.606667   216.097371   0.066923  13.211053  11.415641
8          167    27.330000   168.283280   0.374460   2.969362   7.745036
9          388    32.8